# Workshop-Notebook: Grundlagen der KI-Agenten & Werkzeugnutzung

Dieses Notebook zeigt Ihnen Schritt für Schritt, wie Sie ein Sprachmodell (**Gemini**) mit einem **Werkzeug (Tool)** verbinden und eine einfache **ReAct-Schleife (Orchestrierung)** aufbauen.

**Event:** Digitaltag 2026 | *Mittelstand-Digital Zentrum Spreeland*

### Schritt 1: Installation & Setup
Zuerst installieren wir das offizielle Google GenAI SDK und konfigurieren unseren API-Schlüssel.

**API-Key einrichten:**
Sie benötigen einen kostenlosen Gemini API-Schlüssel von [Google AI Studio](https://aistudio.google.com/app/api-keys).
In Google Colab gibt es zwei Möglichkeiten, den Schlüssel sicher zu hinterlegen:
1. **Interaktiv (Empfohlen für Präsentationen):** Wenn Sie die Zelle unten ausführen, werden Sie nach Ihrem Schlüssel gefragt. Die Eingabe wird maskiert.
2. **Colab-Geheimnisse (Secrets):** Klicken Sie links auf das Schlüssel-Symbol (🔑), fügen Sie ein neues Geheimnis namens `GEMINI_API_KEY` hinzu, fügen Sie Ihren API-Key ein und aktivieren Sie den Schalter "Notebook-Zugriff".

In [ ]:
# Installation des SDKs
!pip install google-genai

In [ ]:
import os
import getpass
from google import genai
from google.genai import types
from google.genai.errors import APIError

# 1. Versuche den Key aus den Colab-Secrets zu laden (falls dort hinterlegt)
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

# 2. Falls nicht gesetzt, frage den Nutzer interaktiv nach dem Key
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Bitte fügen Sie Ihren Gemini API-Key ein und drücken Sie Enter: ")

# 3. Initialisierung des GenAI Clients
client = genai.Client()
model_id = 'gemini-2.5-flash'
print("Gemini API-Schlüssel wurde erfolgreich konfiguriert und der Client initialisiert!")

### Schritt 2: Einfache Modellanfrage
Bevor wir mit Werkzeugen (Tools) arbeiten, testen wir, ob wir das Sprachmodell direkt anfragen können. Wir stellen eine einfache Frage wie: *"Was ist ein KI-Agent?"*

In [ ]:
# Einfache Anfrage an Gemini senden
response = client.models.generate_content(
    model=model_id,
    contents="Was ist ein KI-Agent? Beantworten Sie in 2 bis 3 kurzen Sätzen."
)

print("Antwort von Gemini:\n")
print(response.text)

### Schritt 3: Werkzeug definieren (Tool)
Nun wollen wir dem Modell Zugriff auf Echtzeit-Informationen geben. Wir definieren eine einfache Python-Funktion, die das Wetter für Cottbus abfragt. Diese Funktion wird dem Sprachmodell als Werkzeug übergeben.

In [ ]:
def get_wetter_cottbus(datum: str) -> str:
    """Fragt das aktuelle Wetter für Cottbus an einem bestimmten Datum ab.
    
    Args:
        datum: Das gewünschte Datum im Format JJJJ-MM-TT
    """
    # In der Praxis würde hier eine echte Wetter-API aufgerufen werden
    if datum == "2026-06-26":
        return "Cottbus: 24 Grad, sonnig, Wind aus Nord-Ost, Regenwahrscheinlichkeit 5%."
    else:
        return f"Für das Datum {datum} liegen leider keine Wetterdaten vor. Standardvorhersage: Leicht bewölkt, 18 Grad."

### Schritt 4: Modell mit Werkzeug verbinden (Tool Binding)
Wir senden eine Anfrage, die Wetterdaten benötigt. Das Modell entscheidet selbst, ob es das Werkzeug aufrufen muss. Wenn ja, liefert es uns einen Funktionsaufruf (Function Call) zurück, den wir ausführen müssten.

In [ ]:
response = client.models.generate_content(
    model=model_id,
    contents="Brauche ich am 26. Juni 2026 in Cottbus eine Regenjacke?",
    config=types.GenerateContentConfig(
        # Hier registrieren wir unsere Funktion als Werkzeug
        tools=[get_wetter_cottbus],
        temperature=0
    )
)

# Das Modell führt die Funktion nicht selbst aus, sondern fordert uns dazu auf:
print("Funktionsaufrufe des Modells:", response.function_calls)

### Schritt 5: Die Orchestrierungsschleife (ReAct Loop)
Wenn das Modell entscheidet, das Werkzeug aufzurufen, müssen wir das Ergebnis ausführen und an das Modell zurücksenden. Mit dem SDK können wir das automatische Ausführen (Automatic Function Calling) aktivieren, damit diese Schleife automatisch im Hintergrund abläuft.

In [ ]:
# Das SDK führt die Funktion automatisch aus und liefert die fertige Antwort des Agenten:
response_auto = client.models.generate_content(
    model=model_id,
    contents="Brauche ich am 26. Juni 2026 in Cottbus eine Regenjacke? Beantworte basierend auf dem Wetterbericht.",
    config=types.GenerateContentConfig(
        tools=[get_wetter_cottbus],
        automatic_function_calling=types.AutomaticFunctionCallingConfig(max_remote_calls=3)
    )
)

print("Endgültige Antwort des Agenten:\n")
print(response_auto.text)

### Schritt 6: Höhere Abstraktion mit dem Google Antigravity SDK (Optional)
Für komplexere Agenten (z. B. mit autonomem Dateizugriff, Shell-Ausführung, Richtlinien oder Multi-Agenten-Orchestrierung) bietet Google das **Google Antigravity SDK** (`google-antigravity`) an.

Hier installieren wir das SDK und bauen einen einfachen Agenten, der in einer sicheren Umgebung (mit einer Richtlinie, die alle Aktionen erlaubt) ausgeführt wird.

In [ ]:
# 1. Google Antigravity SDK installieren
!pip install google-antigravity

import asyncio
import os
from google.antigravity import Agent, LocalAgentConfig
from google.antigravity.hooks import policy

# 2. Agenten konfigurieren
# Wir erlauben dem Agenten Werkzeuge autonom auszuführen (policy.allow_all()).
config = LocalAgentConfig(
    system_instructions="Du bist ein hilfreicher KI-Assistent.",
    policies=[policy.allow_all()],
    api_key=os.environ.get("GEMINI_API_KEY")
)

# 3. Den Agenten in einer asynchronen Schleife ausführen
async def run_agent():
    async with Agent(config) as agent:
        response = await agent.chat("Schreibe ein kurzes Gedicht über Cottbus und die Lausitz.")
        text_content = await response.text()
        print("Antwort des Agenten:\n")
        print(text_content)

# In Jupyter / Colab läuft bereits eine Event-Loop.
# Daher führen wir die asynchrone Funktion direkt mit `await` aus:
await run_agent()